# Prediksi Customer Churn dengan TFX Pipeline (Apache Beam), Cloud, dan Monitoring

**Nama:** Fauzi Fadhlurrohman
**Dataset:** [Telco Customer Churn (IBM)](https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv)

Notebook ini membangun *machine learning pipeline* dengan **TensorFlow Extended (TFX)** untuk memprediksi apakah pelanggan layanan telekomunikasi akan berhenti berlangganan (*customer churn*).

Seluruh komponen dijalankan dengan **Apache Beam** sebagai *pipeline orchestrator* melalui `BeamDagRunner` (`from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner`), dan artefaknya disimpan di folder **fauzi19-pipeline**: `CsvExampleGen`, `StatisticsGen`, `SchemaGen`, `ExampleValidator`, `Transform`, `Tuner`, `Trainer`, `Resolver`, `Evaluator`, dan `Pusher`. Komponen didefinisikan pada sel-sel berikut, lalu dirangkai menjadi satu `Pipeline` dan dijalankan oleh `BeamDagRunner` pada bagian 14.

Model hasil pipeline dijalankan di **cloud (Google Cloud Compute Engine)** dan dipantau dengan **Prometheus**; lihat bagian 19 dan 20.

In [1]:
import os
import subprocess
import sys

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

need_install = False
try:
    import tfx
    import pyarrow.lib as pa_lib
    assert hasattr(pa_lib, "PyExtensionType")
except Exception:
    need_install = True

if need_install:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "tfx==1.21.0", "pyarrow==18.1.0", "--force-reinstall",
    ])
    print("Paket TFX dan pyarrow sudah diinstal.")
    print("Jika impor berikutnya gagal, restart runtime lalu jalankan sel ini lagi.")
else:
    print("TFX dan pyarrow siap digunakan.")

TFX dan pyarrow siap digunakan.


## 1. Dataset dan masalah

Dataset yang digunakan adalah **Telco Customer Churn** dari IBM. Dataset ini berisi data pelanggan perusahaan telekomunikasi dengan 21 kolom awal: identitas pelanggan, layanan yang digunakan, kontrak, biaya bulanan, total biaya, dan status churn.

- Jumlah data awal: **7.043 baris**.
- Kolom yang digunakan: **20 kolom** setelah `customerID` dihapus karena hanya berupa identitas.
- Label/target: **Churn** (`Yes` atau `No`).
- Kolom `TotalCharges` memiliki nilai kosong pada pelanggan baru; nilai tersebut dikonversi menjadi numerik dan diisi 0.

**Masalah:** pelanggan yang berhenti berlangganan mengurangi pendapatan perusahaan. **Solusi:** model klasifikasi biner yang memprediksi probabilitas seorang pelanggan akan churn, sehingga tim retensi dapat memprioritaskan pelanggan berisiko.

In [2]:
import os
import pandas as pd

URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

df = pd.read_csv(URL)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0.0)
df = df.drop(columns=["customerID"])
df.to_csv(os.path.join(DATA_DIR, "telco_churn.csv"), index=False)

print("Bentuk data:", df.shape)
print("Nilai kosong:", int(df.isna().sum().sum()))
print(df["Churn"].value_counts(normalize=True).round(4).to_string())
df.head()

Bentuk data: (7043, 20)
Nilai kosong: 0
Churn
No     0.7346
Yes    0.2654


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Modul Transform, Trainer, dan Tuner

Tiga modul Python dipakai oleh komponen TFX dan disimpan sebagai berkas terpisah:

- **fauzi19_transform.py**: `preprocessing_fn` untuk TensorFlow Transform. Fitur numerik distandardisasi dengan z-score, fitur kategorikal dipetakan menjadi indeks vocabulary, dan label `Yes` diubah menjadi 1.
- **fauzi19_tuner.py**: `tuner_fn` untuk komponen Tuner. Keras Tuner (`RandomSearch`) mencari jumlah unit, *dropout rate*, dan *learning rate* terbaik berdasarkan `val_auc`.
- **fauzi19_trainer.py**: `run_fn` untuk komponen Trainer. Model Keras dilatih dengan hyperparameter terbaik dari Tuner, lalu disimpan dengan signature `serving_default` (menerima `tf.Example` mentah dan menerapkan Transform) serta `transform_features` (dipakai Evaluator).

In [3]:
%%writefile fauzi19_transform.py
"""Modul Transform: preprocessing_fn untuk dataset Telco Customer Churn."""

import tensorflow as tf
import tensorflow_transform as tft

LABEL_KEY = "Churn"
NUMERIC_FEATURES = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAL_FEATURES = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod",
]


def _flatten(x):
    """Mengubah SparseTensor atau tensor 2D menjadi tensor 1D."""
    if isinstance(x, tf.SparseTensor):
        x = tf.sparse.to_dense(x)
    return tf.reshape(x, [-1])


def preprocessing_fn(inputs):
    """Standardisasi fitur numerik, vocabulary fitur kategorikal, dan label biner."""
    inputs = {key: _flatten(value) for key, value in inputs.items()}
    outputs = {}

    for key in NUMERIC_FEATURES:
        outputs[f"{key}_xf"] = tft.scale_to_z_score(tf.cast(inputs[key], tf.float32))

    for key in CATEGORICAL_FEATURES:
        outputs[f"{key}_xf"] = tft.compute_and_apply_vocabulary(
            inputs[key], num_oov_buckets=1, vocab_filename=key
        )

    outputs[f"{LABEL_KEY}_xf"] = tf.cast(
        tf.equal(inputs[LABEL_KEY], tf.constant("Yes", dtype=tf.string)), tf.int64
    )
    return outputs

Overwriting fauzi19_transform.py


In [4]:
%%writefile fauzi19_tuner.py
"""Modul Tuner: pencarian hyperparameter otomatis dengan Keras Tuner."""

import keras_tuner as kt
import tensorflow as tf
import tensorflow_transform as tft
from tfx.components.tuner.component import TunerFnResult
from tfx.components.trainer.fn_args_utils import FnArgs

LABEL_KEY = "Churn"
LABEL_TRANSFORMED = f"{LABEL_KEY}_xf"
NUMERIC_FEATURES = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAL_FEATURES = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod",
]
NUMERIC_TRANSFORMED = [f"{key}_xf" for key in NUMERIC_FEATURES]
CATEGORICAL_TRANSFORMED = [f"{key}_xf" for key in CATEGORICAL_FEATURES]
VOCAB_SIZE = 100
EMBEDDING_DIM = 8


def _input_fn(file_pattern, tf_transform_output, batch_size=64, repeat=True):
    """Membaca TFRecord hasil Transform menjadi dataset (fitur, label)."""
    feature_spec = tf_transform_output.transformed_feature_spec()
    if isinstance(file_pattern, str):
        file_pattern = [file_pattern]
    files = []
    for pattern in file_pattern:
        matched = tf.io.gfile.glob(pattern)
        files.extend(matched if matched else [pattern])
    dataset = tf.data.TFRecordDataset(files, compression_type="GZIP")

    def _parse(serialized):
        parsed = tf.io.parse_example(serialized, feature_spec)
        label = parsed.pop(LABEL_TRANSFORMED)
        for key in parsed:
            parsed[key] = tf.expand_dims(parsed[key], -1)
        label = tf.expand_dims(tf.cast(label, tf.float32), -1)
        return parsed, label

    dataset = dataset.map(_parse, num_parallel_calls=tf.data.AUTOTUNE)
    if repeat:
        dataset = dataset.repeat()
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def _build_model(hp):
    """Model yang sama dengan Trainer, dengan hyperparameter yang dicari Tuner."""
    units = hp.Choice("units", [32, 64, 128])
    dropout_rate = hp.Choice("dropout_rate", [0.1, 0.2, 0.3])
    learning_rate = hp.Choice("learning_rate", [0.01, 0.001, 0.0005])

    numeric_inputs = [
        tf.keras.Input(shape=(1,), name=key, dtype=tf.float32) for key in NUMERIC_TRANSFORMED
    ]
    categorical_inputs = [
        tf.keras.Input(shape=(1,), name=key, dtype=tf.int64) for key in CATEGORICAL_TRANSFORMED
    ]

    numeric = tf.keras.layers.Concatenate()(numeric_inputs)
    numeric = tf.keras.layers.Dense(units, activation="relu")(numeric)
    numeric = tf.keras.layers.Dropout(dropout_rate)(numeric)
    numeric = tf.keras.layers.Dense(max(units // 2, 8), activation="relu")(numeric)

    embeddings = []
    for inp in categorical_inputs:
        emb = tf.keras.layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM)(inp)
        embeddings.append(tf.keras.layers.Flatten()(emb))
    categorical = tf.keras.layers.Concatenate()(embeddings)
    categorical = tf.keras.layers.Dense(max(units // 2, 8), activation="relu")(categorical)

    x = tf.keras.layers.Concatenate()([numeric, categorical])
    x = tf.keras.layers.Dense(max(units // 2, 8), activation="relu")(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    output = tf.keras.layers.Dense(1, activation="sigmoid", name="churn_probability")(x)

    model = tf.keras.Model(inputs=numeric_inputs + categorical_inputs, outputs=output)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="binary_accuracy"),
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )
    return model


def tuner_fn(fn_args: FnArgs) -> TunerFnResult:
    """Entry point Tuner: RandomSearch dengan objective val_auc."""
    tf_transform_output = tft.TFTransformOutput(fn_args.transform_graph_path)
    train_dataset = _input_fn(fn_args.train_files, tf_transform_output, repeat=True)
    eval_dataset = _input_fn(fn_args.eval_files, tf_transform_output, repeat=True)

    tuner = kt.RandomSearch(
        _build_model,
        objective=kt.Objective("val_auc", direction="max"),
        max_trials=6,
        seed=42,
        directory=fn_args.working_dir,
        project_name="churn_tuning",
    )
    return TunerFnResult(
        tuner=tuner,
        fit_kwargs={
            "x": train_dataset,
            "validation_data": eval_dataset,
            "steps_per_epoch": fn_args.train_steps,
            "validation_steps": fn_args.eval_steps,
            "epochs": 2,
        },
    )

Overwriting fauzi19_tuner.py


In [5]:
%%writefile fauzi19_trainer.py
"""Modul Trainer: model Keras untuk prediksi churn, memakai hyperparameter hasil Tuner."""

import keras
import keras_tuner as kt
import tensorflow as tf
import tensorflow_transform as tft
from tfx.components.trainer.fn_args_utils import FnArgs

LABEL_KEY = "Churn"
LABEL_TRANSFORMED = f"{LABEL_KEY}_xf"
NUMERIC_FEATURES = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAL_FEATURES = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod",
]
NUMERIC_TRANSFORMED = [f"{key}_xf" for key in NUMERIC_FEATURES]
CATEGORICAL_TRANSFORMED = [f"{key}_xf" for key in CATEGORICAL_FEATURES]
VOCAB_SIZE = 100
EMBEDDING_DIM = 8

# Hyperparameter bawaan bila Trainer dijalankan tanpa hasil Tuner.
DEFAULT_HYPERPARAMETERS = {"units": 64, "dropout_rate": 0.2, "learning_rate": 0.001}


def _input_fn(file_pattern, tf_transform_output, batch_size=64, repeat=True):
    """Membaca TFRecord hasil Transform menjadi dataset (fitur, label)."""
    feature_spec = tf_transform_output.transformed_feature_spec()
    if isinstance(file_pattern, str):
        file_pattern = [file_pattern]
    files = []
    for pattern in file_pattern:
        matched = tf.io.gfile.glob(pattern)
        files.extend(matched if matched else [pattern])
    dataset = tf.data.TFRecordDataset(files, compression_type="GZIP")

    def _parse(serialized):
        parsed = tf.io.parse_example(serialized, feature_spec)
        label = parsed.pop(LABEL_TRANSFORMED)
        for key in parsed:
            parsed[key] = tf.expand_dims(parsed[key], -1)
        label = tf.expand_dims(tf.cast(label, tf.float32), -1)
        return parsed, label

    dataset = dataset.map(_parse, num_parallel_calls=tf.data.AUTOTUNE)
    if repeat:
        dataset = dataset.repeat()
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def _hyperparameter(hparams, name):
    """Mengambil nilai hyperparameter dari hasil Tuner atau nilai bawaan."""
    if hparams is not None and name in hparams.values:
        return hparams.get(name)
    return DEFAULT_HYPERPARAMETERS[name]


def build_model(hparams=None):
    """Membangun model Keras dua cabang (numerik dan kategorikal)."""
    units = int(_hyperparameter(hparams, "units"))
    dropout_rate = float(_hyperparameter(hparams, "dropout_rate"))
    learning_rate = float(_hyperparameter(hparams, "learning_rate"))

    numeric_inputs = [
        tf.keras.Input(shape=(1,), name=key, dtype=tf.float32) for key in NUMERIC_TRANSFORMED
    ]
    categorical_inputs = [
        tf.keras.Input(shape=(1,), name=key, dtype=tf.int64) for key in CATEGORICAL_TRANSFORMED
    ]

    numeric = tf.keras.layers.Concatenate()(numeric_inputs)
    numeric = tf.keras.layers.Dense(units, activation="relu")(numeric)
    numeric = tf.keras.layers.Dropout(dropout_rate)(numeric)
    numeric = tf.keras.layers.Dense(max(units // 2, 8), activation="relu")(numeric)

    embeddings = []
    for inp in categorical_inputs:
        emb = tf.keras.layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM)(inp)
        embeddings.append(tf.keras.layers.Flatten()(emb))
    categorical = tf.keras.layers.Concatenate()(embeddings)
    categorical = tf.keras.layers.Dense(max(units // 2, 8), activation="relu")(categorical)

    x = tf.keras.layers.Concatenate()([numeric, categorical])
    x = tf.keras.layers.Dense(max(units // 2, 8), activation="relu")(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    output = tf.keras.layers.Dense(1, activation="sigmoid", name="churn_probability")(x)

    model = tf.keras.Model(inputs=numeric_inputs + categorical_inputs, outputs=output)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="binary_accuracy"),
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )
    return model


def _export_for_serving(model, tf_transform_output, export_dir):
    """Mengekspor SavedModel yang kompatibel dengan TensorFlow Serving (Keras 3 ExportArchive).

    Dua signature dibuat:
    - serving_default: menerima tf.Example serial, menerapkan Transform, lalu memprediksi.
    - transform_features: dipakai Evaluator untuk menyiapkan fitur dan label.
    """
    tft_layer = tf_transform_output.transform_features_layer()
    input_signature = [tf.TensorSpec(shape=[None], dtype=tf.string, name="examples")]

    @tf.function(input_signature=input_signature)
    def serve_tf_examples_fn(serialized_tf_examples):
        raw_feature_spec = tf_transform_output.raw_feature_spec()
        raw_feature_spec.pop(LABEL_KEY)
        raw_features = tf.io.parse_example(serialized_tf_examples, raw_feature_spec)
        transformed_features = tft_layer(raw_features)
        return {"outputs": model(transformed_features)}

    @tf.function(input_signature=input_signature)
    def transform_features_fn(serialized_tf_example):
        raw_feature_spec = tf_transform_output.raw_feature_spec()
        raw_features = tf.io.parse_example(serialized_tf_example, raw_feature_spec)
        return tft_layer(raw_features)

    archive = keras.export.ExportArchive()
    archive.track(model)
    archive.track(tft_layer)
    archive.add_endpoint("serving_default", serve_tf_examples_fn)
    archive.add_endpoint("transform_features", transform_features_fn)
    archive.write_out(export_dir)


def run_fn(fn_args: FnArgs):
    """Entry point Trainer."""
    tf_transform_output = tft.TFTransformOutput(fn_args.transform_output)
    train_dataset = _input_fn(fn_args.train_files, tf_transform_output, repeat=True)
    eval_dataset = _input_fn(fn_args.eval_files, tf_transform_output, repeat=True)

    hparams = None
    if fn_args.hyperparameters:
        hparams = kt.HyperParameters.from_config(fn_args.hyperparameters)
        print("Hyperparameter dari Tuner:", hparams.values)
    else:
        print("Tuner tidak digunakan, memakai hyperparameter bawaan:", DEFAULT_HYPERPARAMETERS)

    model = build_model(hparams)
    model.fit(
        train_dataset,
        steps_per_epoch=fn_args.train_steps,
        validation_data=eval_dataset,
        validation_steps=fn_args.eval_steps,
        epochs=3,
        callbacks=[
            tf.keras.callbacks.TensorBoard(log_dir=fn_args.model_run_dir, update_freq="batch"),
            tf.keras.callbacks.EarlyStopping(
                monitor="val_auc", mode="max", patience=2, restore_best_weights=True
            ),
        ],
    )

    _export_for_serving(model, tf_transform_output, fn_args.serving_model_dir)

Overwriting fauzi19_trainer.py


## 3. Inisialisasi pipeline

Pipeline root diarahkan ke **/content/fauzi19-pipeline**. Folder dibersihkan terlebih dahulu agar setiap eksekusi dimulai dari awal. Pipeline orchestrator yang dipakai adalah **Apache Beam** (`BeamDagRunner`), dengan metadata ML Metadata disimpan pada SQLite di dalam pipeline root.

In [6]:
import logging
import os
import shutil

import tensorflow as tf
import tfx
from tfx.components import (
    CsvExampleGen, Evaluator, ExampleValidator, Pusher, SchemaGen,
    StatisticsGen, Trainer, Transform, Tuner,
)
from tfx.components.trainer.executor import GenericExecutor
from tfx.dsl.components.base import executor_spec
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.experimental import latest_blessed_model_resolver
from tfx.orchestration import metadata, pipeline
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner
from tfx.proto import pusher_pb2, trainer_pb2
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing

logging.getLogger("tensorflow").setLevel(logging.ERROR)

PIPELINE_NAME = "fauzi19-pipeline"
PIPELINE_ROOT = "/content/fauzi19-pipeline"
METADATA_PATH = os.path.join(PIPELINE_ROOT, "metadata.sqlite")
SERVING_MODEL_DIR = "/content/serving_model"
TRANSFORM_MODULE = "/content/fauzi19_transform.py"
TUNER_MODULE = "/content/fauzi19_tuner.py"
TRAINER_MODULE = "/content/fauzi19_trainer.py"

for path in (PIPELINE_ROOT, SERVING_MODEL_DIR):
    if os.path.exists(path):
        shutil.rmtree(path)
os.makedirs(PIPELINE_ROOT, exist_ok=True)

print("TFX", tfx.__version__, "| TensorFlow", tf.__version__)
print("Orchestrator: BeamDagRunner (Apache Beam)")
print("Pipeline root:", PIPELINE_ROOT)

TFX 1.21.0 | TensorFlow 2.21.0
Orchestrator: BeamDagRunner (Apache Beam)
Pipeline root: /content/fauzi19-pipeline


## 4. ExampleGen

**CsvExampleGen** membaca berkas CSV bersih dari `/content/data` dan mengubahnya menjadi artefak *Examples* berformat TFRecord. ExampleGen juga membagi data menjadi split **train** dan **eval** dengan konfigurasi bawaan TFX (sekitar 2/3 train dan 1/3 eval), sehingga evaluasi dilakukan pada data yang tidak dipakai untuk melatih model.

In [7]:
example_gen = CsvExampleGen(input_base=DATA_DIR)

## 5. StatisticsGen

**StatisticsGen** menghitung statistik deskriptif setiap fitur, seperti jumlah nilai, minimum/maksimum, rata-rata, dan distribusi kategori. Statistik ini menjadi dasar bagi SchemaGen dan ExampleValidator.

In [8]:
statistics_gen = StatisticsGen(examples=example_gen.outputs["examples"])

## 6. SchemaGen

**SchemaGen** membuat schema data dari statistik: tipe data, domain nilai, dan presensi tiap fitur. Schema dipakai oleh ExampleValidator, Transform, Tuner, dan Trainer.

In [9]:
schema_gen = SchemaGen(
    statistics=statistics_gen.outputs["statistics"],
    infer_feature_shape=False,
)

## 7. ExampleValidator

**ExampleValidator** membandingkan statistik data dengan schema untuk mendeteksi anomali, misalnya nilai hilang, tipe data yang salah, atau nilai di luar domain. Hasilnya disimpan sebagai artefak *anomalies*.

In [10]:
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs["statistics"],
    schema=schema_gen.outputs["schema"],
)

## 8. Transform

**Transform** menjalankan `preprocessing_fn` dari `fauzi19_transform.py`. Fitur numerik distandardisasi dengan z-score, fitur kategorikal dipetakan menjadi indeks vocabulary (dengan satu *out-of-vocabulary bucket*), dan label `Churn` diubah menjadi `Churn_xf` (Yes = 1, No = 0). Transform menghasilkan *transformed examples* untuk pelatihan dan *transform graph* yang dipakai ulang saat evaluasi dan serving, sehingga preprocessing konsisten.

In [11]:
transform = Transform(
    examples=example_gen.outputs["examples"],
    schema=schema_gen.outputs["schema"],
    module_file=TRANSFORM_MODULE,
)

## 9. Tuner

**Tuner** menjalankan pencarian hyperparameter otomatis dengan Keras Tuner (`RandomSearch`, 6 percobaan). Ruang pencarian: jumlah unit (32, 64, 128), *dropout rate* (0,1; 0,2; 0,3), dan *learning rate* (0,01; 0,001; 0,0005). Metrik yang dioptimalkan adalah `val_auc`. Hyperparameter terbaik diteruskan ke Trainer.

In [12]:
tuner = Tuner(
    module_file=TUNER_MODULE,
    examples=transform.outputs["transformed_examples"],
    transform_graph=transform.outputs["transform_graph"],
    schema=schema_gen.outputs["schema"],
    train_args=trainer_pb2.TrainArgs(num_steps=200),
    eval_args=trainer_pb2.EvalArgs(num_steps=50),
)

## 10. Trainer

**Trainer** melatih model Keras dari *transformed examples* dan *transform graph* dengan hyperparameter terbaik dari Tuner. Model menerima empat fitur numerik yang sudah distandardisasi dan lima belas fitur kategorikal yang sudah menjadi indeks vocabulary.

- Cabang numerik: Concatenate, Dense (units) ReLU, Dropout, Dense (units/2) ReLU.
- Cabang kategorikal: Embedding 8 dimensi per fitur, Flatten, Concatenate, Dense (units/2) ReLU.
- Gabungan: Concatenate, Dense (units/2) ReLU, Dropout, Dense 1 sigmoid.
- Optimizer Adam, loss *binary crossentropy*, *early stopping* pada `val_auc`.

Model diekspor dengan `keras.export.ExportArchive` sebagai SavedModel yang kompatibel dengan TensorFlow Serving, dengan signature `serving_default` dan `transform_features`.

In [13]:
trainer = Trainer(
    module_file=TRAINER_MODULE,
    examples=transform.outputs["transformed_examples"],
    transform_graph=transform.outputs["transform_graph"],
    schema=schema_gen.outputs["schema"],
    hyperparameters=tuner.outputs["best_hyperparameters"],
    train_args=trainer_pb2.TrainArgs(num_steps=500),
    eval_args=trainer_pb2.EvalArgs(num_steps=100),
    custom_executor_spec=executor_spec.ExecutorClassSpec(GenericExecutor),
)

## 11. Resolver

**Resolver** mencari model yang sudah mendapat *blessing* dari Evaluator untuk dijadikan baseline. Pada eksekusi pertama belum ada baseline, sehingga output model kosong. Hal ini normal: Evaluator tetap dapat mengevaluasi model pertama tanpa baseline dan memberi blessing bila ambang metrik terpenuhi.

In [14]:
resolver = Resolver(
    strategy_class=latest_blessed_model_resolver.LatestBlessedModelResolver,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing),
).with_id("latest_blessed_model_resolver")

## 12. Evaluator

**Evaluator** mengevaluasi model pada split eval memakai TensorFlow Model Analysis (TFMA). Evaluator menggunakan signature `serving_default` dan memanggil `transform_features` terlebih dahulu sehingga label `Churn_xf` dan fitur dihitung dengan cara yang sama seperti saat training.

Metrik: ExampleCount, BinaryAccuracy, AUC, Precision, dan Recall. Ambang blessing: BinaryAccuracy minimal **0,70**.

In [15]:
import tensorflow_model_analysis as tfma

eval_config = tfma.EvalConfig(
    model_specs=[
        tfma.ModelSpec(
            signature_name="serving_default",
            label_key="Churn_xf",
            preprocessing_function_names=["transform_features"],
        )
    ],
    slicing_specs=[tfma.SlicingSpec()],
    metrics_specs=[
        tfma.MetricsSpec(metrics=[
            tfma.MetricConfig(class_name="ExampleCount"),
            tfma.MetricConfig(
                class_name="BinaryAccuracy",
                threshold=tfma.MetricThreshold(
                    value_threshold=tfma.GenericValueThreshold(lower_bound={"value": 0.70}),
                    change_threshold=tfma.GenericChangeThreshold(
                        direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                        absolute={"value": -1e-10},
                    ),
                ),
            ),
            tfma.MetricConfig(class_name="AUC"),
            tfma.MetricConfig(class_name="Precision"),
            tfma.MetricConfig(class_name="Recall"),
        ])
    ],
)

evaluator = Evaluator(
    examples=example_gen.outputs["examples"],
    model=trainer.outputs["model"],
    baseline_model=resolver.outputs["model"],
    eval_config=eval_config,
    schema=schema_gen.outputs["schema"],
)

## 13. Pusher

**Pusher** menerima model dari Trainer hanya jika model sudah mendapat blessing dari Evaluator. Model yang lolos disalin ke folder **serving_model**, yang siap dipakai TensorFlow Serving.

In [16]:
pusher = Pusher(
    model=trainer.outputs["model"],
    model_blessing=evaluator.outputs["blessing"],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(base_directory=SERVING_MODEL_DIR)
    ),
)

## 14. Menjalankan pipeline dengan Apache Beam (BeamDagRunner)

Seluruh komponen di atas dirangkai menjadi satu objek `Pipeline` (beserta konfigurasi metadata SQLite), lalu dijalankan oleh **`BeamDagRunner`**, *pipeline orchestrator* berbasis **Apache Beam**. BeamDagRunner mengeksekusi komponen sesuai urutan dependensinya: `CsvExampleGen`, `StatisticsGen`, `SchemaGen`, `ExampleValidator`, `Transform`, `Tuner`, `Trainer`, `Resolver`, `Evaluator`, dan `Pusher`. Artefak setiap komponen tersimpan di `/content/fauzi19-pipeline/<NamaKomponen>/...`.

In [17]:
components = [
    example_gen,
    statistics_gen,
    schema_gen,
    example_validator,
    transform,
    tuner,
    trainer,
    resolver,
    evaluator,
    pusher,
]

churn_pipeline = pipeline.Pipeline(
    pipeline_name=PIPELINE_NAME,
    pipeline_root=PIPELINE_ROOT,
    components=components,
    enable_cache=False,
    metadata_connection_config=metadata.sqlite_metadata_connection_config(METADATA_PATH),
)

BeamDagRunner().run(churn_pipeline)
print("\nPipeline selesai dijalankan oleh BeamDagRunner.")

Trial 6 Complete [00h 00m 19s]
val_auc: 0.8510334491729736

Best val_auc So Far: 0.8510334491729736
Total elapsed time: 00h 01m 57s
Results summary
Results in /content/fauzi19-pipeline/Tuner/.system/executor_execution/7/.temp/7/churn_tuning
Showing 10 best trials
Objective(name="val_auc", direction="max")

Trial 5 summary
Hyperparameters:
units: 64
dropout_rate: 0.2
learning_rate: 0.001
Score: 0.8510334491729736

Trial 2 summary
Hyperparameters:
units: 128
dropout_rate: 0.1
learning_rate: 0.0005
Score: 0.8510030508041382

Trial 0 summary
Hyperparameters:
units: 64
dropout_rate: 0.1
learning_rate: 0.001
Score: 0.8501310348510742

Trial 3 summary
Hyperparameters:
units: 32
dropout_rate: 0.2
learning_rate: 0.01
Score: 0.849270224571228

Trial 4 summary
Hyperparameters:
units: 32
dropout_rate: 0.2
learning_rate: 0.0005
Score: 0.8472939729690552

Trial 1 summary
Hyperparameters:
units: 32
dropout_rate: 0.3
learning_rate: 0.001
Score: 0.8434889912605286


Hyperparameter dari Tuner: {'units': 64, 'dropout_rate': 0.2, 'learning_rate': 0.001}
Epoch 1/3
500/500 ━━━━━━━━━━━━━━━━━━━━ 16s 22ms/step - auc: 0.8285 - binary_accuracy: 0.7963 - loss: 0.4349 - precision: 0.6503 - recall: 0.5049 - val_auc: 0.8515 - val_binary_accuracy: 0.8027 - val_loss: 0.4121 - val_precision: 0.6428 - val_recall: 0.5638
Epoch 2/3
500/500 ━━━━━━━━━━━━━━━━━━━━ 9s 19ms/step - auc: 0.8470 - binary_accuracy: 0.8093 - loss: 0.4142 - precision: 0.6765 - recall: 0.5412 - val_auc: 0.8474 - val_binary_accuracy: 0.8034 - val_loss: 0.4124 - val_precision: 0.6629 - val_recall: 0.5157
Epoch 3/3
500/500 ━━━━━━━━━━━━━━━━━━━━ 9s 18ms/step - auc: 0.8521 - binary_accuracy: 0.8128 - loss: 0.4086 - precision: 0.6885 - recall: 0.5414 - val_auc: 0.8450 - val_binary_accuracy: 0.7992 - val_loss: 0.4185 - val_precision: 0.6742 - val_recall: 0.4593
Saved artifact at '/content/fauzi19-pipeline/Trainer/model/8/Format-Serving'. The following endpoints are available:

* Endpoint 'serving_default


Pipeline selesai dijalankan oleh BeamDagRunner.


## 15. Hasil eksekusi komponen

Sel berikut membaca artefak keluaran setiap komponen dari pipeline root (`/content/fauzi19-pipeline`): statistik data, schema, anomali, hasil Transform, hyperparameter terbaik dari Tuner, dan metrik Evaluator.

In [18]:
import json

import pandas as pd
import tensorflow_data_validation as tfdv


def artifact_uri(component, output):
    """Folder artefak terbaru milik sebuah komponen pada pipeline root."""
    base = os.path.join(PIPELINE_ROOT, component, output)
    execution_ids = sorted((int(name) for name in os.listdir(base) if name.isdigit()))
    return os.path.join(base, str(execution_ids[-1]))


for component, output in [
    ("CsvExampleGen", "examples"), ("StatisticsGen", "statistics"), ("SchemaGen", "schema"),
    ("ExampleValidator", "anomalies"), ("Transform", "transform_graph"), ("Tuner", "best_hyperparameters"),
    ("Trainer", "model"), ("Evaluator", "evaluation"), ("Evaluator", "blessing"), ("Pusher", "pushed_model"),
]:
    print(f"{component:17s} {output:21s}", artifact_uri(component, output))

CsvExampleGen     examples              /content/fauzi19-pipeline/CsvExampleGen/examples/2
StatisticsGen     statistics            /content/fauzi19-pipeline/StatisticsGen/statistics/3
SchemaGen         schema                /content/fauzi19-pipeline/SchemaGen/schema/4
ExampleValidator  anomalies             /content/fauzi19-pipeline/ExampleValidator/anomalies/5
Transform         transform_graph       /content/fauzi19-pipeline/Transform/transform_graph/6
Tuner             best_hyperparameters  /content/fauzi19-pipeline/Tuner/best_hyperparameters/7
Trainer           model                 /content/fauzi19-pipeline/Trainer/model/8
Evaluator         evaluation            /content/fauzi19-pipeline/Evaluator/evaluation/9
Evaluator         blessing              /content/fauzi19-pipeline/Evaluator/blessing/9
Pusher            pushed_model          /content/fauzi19-pipeline/Pusher/pushed_model/10


### Statistik data (StatisticsGen)

In [19]:
stats_uri = artifact_uri("StatisticsGen", "statistics")
train_stats = tfdv.load_stats_binary(os.path.join(stats_uri, "Split-train", "FeatureStats.pb"))
eval_stats = tfdv.load_stats_binary(os.path.join(stats_uri, "Split-eval", "FeatureStats.pb"))
print("Jumlah contoh train:", train_stats.datasets[0].num_examples)
print("Jumlah contoh eval :", eval_stats.datasets[0].num_examples)
tfdv.visualize_statistics(lhs_statistics=eval_stats, rhs_statistics=train_stats,
                          lhs_name="eval", rhs_name="train")

Jumlah contoh train: 4706
Jumlah contoh eval : 2337


### Schema (SchemaGen)

In [20]:
schema = tfdv.load_schema_text(os.path.join(artifact_uri("SchemaGen", "schema"), "schema.pbtxt"))
tfdv.display_schema(schema)

,Type,Presence,Valency,Domain
Feature name,,,,
'Churn',STRING,required,single,'Churn'
'Contract',STRING,required,single,'Contract'
'Dependents',STRING,required,single,'Dependents'
'DeviceProtection',STRING,required,single,'DeviceProtection'
'InternetService',STRING,required,single,'InternetService'
'MonthlyCharges',FLOAT,required,single,-
'MultipleLines',STRING,required,single,'MultipleLines'
'OnlineBackup',STRING,required,single,'OnlineBackup'
'OnlineSecurity',STRING,required,single,'OnlineSecurity'


,Values
Domain,
'Churn',"'No', 'Yes'"
'Contract',"'Month-to-month', 'One year', 'Two year'"
'Dependents',"'No', 'Yes'"
'DeviceProtection',"'No', 'No internet service', 'Yes'"
'InternetService',"'DSL', 'Fiber optic', 'No'"
'MultipleLines',"'No', 'No phone service', 'Yes'"
'OnlineBackup',"'No', 'No internet service', 'Yes'"
'OnlineSecurity',"'No', 'No internet service', 'Yes'"
'PaperlessBilling',"'No', 'Yes'"


### Anomali (ExampleValidator)

In [21]:
from tensorflow_metadata.proto.v0 import anomalies_pb2

anomalies_uri = artifact_uri("ExampleValidator", "anomalies")
for split in ("Split-train", "Split-eval"):
    anomalies = anomalies_pb2.Anomalies()
    with open(os.path.join(anomalies_uri, split, "SchemaDiff.pb"), "rb") as f:
        anomalies.ParseFromString(f.read())
    print(split, "-> jumlah anomali:", len(anomalies.anomaly_info))
tfdv.display_anomalies(anomalies)

Split-train -> jumlah anomali: 0
Split-eval -> jumlah anomali: 0


### Hasil Transform

In [22]:
import tensorflow_transform as tft

transform_graph_uri = artifact_uri("Transform", "transform_graph")
tf_transform_output = tft.TFTransformOutput(transform_graph_uri)
print("Jumlah fitur hasil Transform:", len(tf_transform_output.transformed_feature_spec()))
print(sorted(tf_transform_output.transformed_feature_spec().keys()))

Jumlah fitur hasil Transform: 20
['Churn_xf', 'Contract_xf', 'Dependents_xf', 'DeviceProtection_xf', 'InternetService_xf', 'MonthlyCharges_xf', 'MultipleLines_xf', 'OnlineBackup_xf', 'OnlineSecurity_xf', 'PaperlessBilling_xf', 'Partner_xf', 'PaymentMethod_xf', 'PhoneService_xf', 'SeniorCitizen_xf', 'StreamingMovies_xf', 'StreamingTV_xf', 'TechSupport_xf', 'TotalCharges_xf', 'gender_xf', 'tenure_xf']


### Hyperparameter terbaik (Tuner)

In [23]:
import json

best_hp_uri = artifact_uri("Tuner", "best_hyperparameters")
with open(os.path.join(best_hp_uri, "best_hyperparameters.txt")) as f:
    best_hp = json.load(f)
print("Hyperparameter terbaik hasil Tuner:")
print(json.dumps(best_hp["values"], indent=2))

Hyperparameter terbaik hasil Tuner:
{
  "units": 64,
  "dropout_rate": 0.2,
  "learning_rate": 0.001
}


### Metrik model (Evaluator)

In [24]:
import tensorflow_model_analysis as tfma

eval_result = tfma.load_eval_result(artifact_uri("Evaluator", "evaluation"))
metrics = eval_result.get_metrics_for_slice()
metrics_table = pd.DataFrame(
    {"nilai": {name: round(value["doubleValue"], 4) for name, value in metrics.items()}}
)
blessing_uri = artifact_uri("Evaluator", "blessing")
print("Model mendapat blessing:", os.path.exists(os.path.join(blessing_uri, "BLESSED")))
metrics_table

Model mendapat blessing: True


,nilai
example_count,2337.0000
binary_accuracy,0.8023
auc,0.8512
precision,0.6455
recall,0.5599


## 16. Verifikasi artefak

Sel berikut memeriksa bahwa seluruh komponen sudah berjalan dan artefaknya berada di folder **fauzi19-pipeline**, serta model hasil Pusher dapat dimuat kembali.

In [25]:
expected = [
    "CsvExampleGen", "StatisticsGen", "SchemaGen", "ExampleValidator",
    "Transform", "Tuner", "Trainer", "Evaluator", "Pusher",
]
print("Pipeline root:", PIPELINE_ROOT)
for name in expected:
    print(f"{name:18s} ada:", os.path.isdir(os.path.join(PIPELINE_ROOT, name)))

versions = sorted(os.listdir(SERVING_MODEL_DIR))
print("Versi serving model:", versions)
loaded = tf.saved_model.load(os.path.join(SERVING_MODEL_DIR, versions[-1]))
print("Signature serving model:", sorted(loaded.signatures.keys()))
print("Total file artefak pipeline:", sum(len(files) for _, _, files in os.walk(PIPELINE_ROOT)))

Pipeline root: /content/fauzi19-pipeline
CsvExampleGen      ada: True
StatisticsGen      ada: True
SchemaGen          ada: True
ExampleValidator   ada: True
Transform          ada: True
Tuner              ada: True
Trainer            ada: True
Evaluator          ada: True
Pusher             ada: True
Versi serving model: ['1793092663']
Signature serving model: ['serving_default', 'transform_features']
Total file artefak pipeline: 109


## 17. requirements.txt

Dependensi dikumpulkan dari seluruh `import` pada modul Python dan notebook ini, lalu dipetakan ke nama paket beserta versi yang benar-benar terpasang pada runtime. Hasilnya disimpan sebagai `requirements.txt`.

In [26]:
import ast
from importlib import metadata

modules = [TRANSFORM_MODULE, TUNER_MODULE, TRAINER_MODULE]
imported = set()
for path in modules:
    with open(path) as f:
        tree = ast.parse(f.read())
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            imported.update(alias.name.split(".")[0] for alias in node.names)
        elif isinstance(node, ast.ImportFrom) and node.module and node.level == 0:
            imported.add(node.module.split(".")[0])

# Impor yang dipakai langsung pada sel notebook ini.
imported.update({
    "numpy", "pandas", "requests", "tensorflow",
    "tensorflow_data_validation", "tensorflow_model_analysis", "tensorflow_transform", "tfx",
})
imported = {name for name in imported if name not in sys.stdlib_module_names}

owners = metadata.packages_distributions()
requirements = {}
for name in sorted(imported):
    for dist in owners.get(name, []):
        requirements[dist] = metadata.version(dist)
requirements["pyarrow"] = metadata.version("pyarrow")

with open("/content/requirements.txt", "w") as f:
    for dist in sorted(requirements, key=str.lower):
        f.write(f"{dist}=={requirements[dist]}\n")

print(open("/content/requirements.txt").read())

keras==3.15.1
keras-tuner==1.4.8
ml-pipelines-sdk==1.21.0
numpy==2.4.6
pandas==2.3.3
pyarrow==18.1.0
requests==2.34.2
tensorflow==2.21.0
tensorflow-data-validation==1.21.0
tensorflow-transform==1.21.0
tensorflow_model_analysis==0.52.0
tfx==1.21.0



## 18. Performa model

Nilai berikut diambil dari output komponen **Evaluator** pada split eval (2.337 contoh), seperti pada sel metrik model di bagian 15.

| Metrik | Nilai |
| ----------- | ----------- |
| AUC | 0,8512 |
| Binary Accuracy | 0,8023 |
| Precision | 0,6455 |
| Recall | 0,5599 |

Model mendapat *blessing* dari Evaluator (Binary Accuracy minimal 0,70) dan dipublikasikan oleh Pusher ke folder `serving_model`. Hyperparameter terbaik hasil Tuner: units 64, dropout rate 0,2, learning rate 0,001 (`val_auc` Tuner 0,8510).

## 19. Deployment ke Cloud

### Opsi deployment dan platform

| Opsi | Pertimbangan |
| ----------- | ----------- |
| Batch prediction | Cocok bila churn dihitung berkala (mingguan), tetapi tim retensi tidak dapat memeriksa pelanggan secara langsung. |
| **Online serving lewat REST API (dipilih)** | Prediksi real-time per pelanggan dan mudah diintegrasikan ke aplikasi CRM. |
| Platform | **Google Cloud Platform, Compute Engine (VM e2-micro, region us-central1-a)**. VM berada pada *free tier* sehingga tanpa biaya, dan dikelola dengan `systemd` agar layanan hidup kembali otomatis. |

### Arsitektur

```text
Klien --HTTP :8080--> Gateway (Flask + gunicorn) --REST :8501--> TensorFlow Serving --> model hasil Pusher
                           |  /metrics                              |  /monitoring/prometheus/metrics
                           +----------------> Prometheus (:9090, scrape 15 detik) <-----+
```

- **TensorFlow Serving** (`tensorflow_model_server` 2.21.0 dari paket apt) memuat `serving_model/1793092663` hasil komponen Pusher (`deployment/fauzi19-tfserving.service`, memakai `monitoring/prometheus.config`). Image Docker setara tersedia pada `Dockerfile`.
- **Gateway** (`deployment/gateway.py`) menerima fitur pelanggan dalam JSON, memvalidasi nilai, membentuk `tf.Example`, memanggil TensorFlow Serving, lalu mengembalikan probabilitas dan label. Gateway juga menyediakan halaman web untuk mencoba prediksi dan mengekspor metrik Prometheus.
- **Prometheus** 3.5.0 memantau gateway dan TensorFlow Serving (lihat bagian Monitoring).
- Firewall hanya membuka port 8080. Endpoint metrik TensorFlow Serving diteruskan gateway pada `http://34.122.18.16:8080/monitoring/prometheus/metrics`, dan Prometheus mengambilnya dari URL cloud tersebut (bukan localhost). Port 9090 Prometheus dan 8501 TensorFlow Serving hanya dapat diakses dari VM.
- Instalasi: `deployment/setup.sh` memasang TensorFlow Serving, virtualenv gateway, Prometheus, dan mendaftarkan ketiga layanan `systemd`.

### Tautan web app

- **Web app**: http://34.122.18.16:8080/ (formulir prediksi churn)
- **Endpoint API**: `POST http://34.122.18.16:8080/predict`
- Kesehatan layanan: http://34.122.18.16:8080/health
- Metrik Prometheus gateway: http://34.122.18.16:8080/metrics
- Metrik Prometheus TensorFlow Serving (endpoint yang di-scrape Prometheus): http://34.122.18.16:8080/monitoring/prometheus/metrics
- Skema fitur: http://34.122.18.16:8080/schema
- **Status model TensorFlow Serving (cloud)**: http://34.122.18.16:8080/v1/models/fauzi19-churn-model
- **Metadata model TensorFlow Serving (cloud)**: http://34.122.18.16:8080/v1/models/fauzi19-churn-model/metadata

Bukti serving berjalan di cloud: `fauzi19-deployment.png` adalah tangkapan layar browser (address bar terlihat) yang mengakses `http://34.122.18.16:8080/v1/models/fauzi19-churn-model/metadata` pada VM Google Cloud.

Contoh request:

```bash
curl -X POST http://34.122.18.16:8080/predict -H "Content-Type: application/json" -d '{"gender":"Female","SeniorCitizen":0,"Partner":"No","Dependents":"No","tenure":2,"PhoneService":"Yes","MultipleLines":"No","InternetService":"Fiber optic","OnlineSecurity":"No","OnlineBackup":"No","DeviceProtection":"No","TechSupport":"No","StreamingTV":"No","StreamingMovies":"No","Contract":"Month-to-month","PaperlessBilling":"Yes","PaymentMethod":"Electronic check","MonthlyCharges":80.35,"TotalCharges":160.7}'
# {"label":"churn","probability":0.736}
```

Mengirim *list* JSON berisi beberapa pelanggan akan mengembalikan `{"predictions": [...]}`.

### Pengujian

`fauzi19-testing.ipynb` menguji endpoint cloud di atas: satu pelanggan berisiko (probabilitas churn 0,736) dan satu pelanggan aman (0,0121), batch 200 pelanggan (accuracy 0,7900, precision 0,6596, recall 0,5439; sejalan dengan metrik Evaluator pipeline), serta validasi input (422).



In [1]:
import requests

BASE_URL = "http://34.122.18.16:8080"
print(requests.get(BASE_URL + "/health", timeout=30).json())

customer = {
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "No",
    "Dependents": "No",
    "tenure": 2,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 80.35,
    "TotalCharges": 160.7
}
response = requests.post(BASE_URL + "/predict", json=customer, timeout=30)
print(response.status_code, response.json())

{'status': 'ok'}
200 {'label': 'churn', 'probability': 0.736}


## 20. Monitoring dengan Prometheus

Prometheus berjalan di VM yang sama dan dikonfigurasi lewat direktori `monitoring/` (`Dockerfile`, `prometheus.yml`, dan `prometheus.config`; port 9090 hanya untuk localhost). Label global `monitor="tf-serving-monitor"`; TensorFlow Serving di-*scrape* setiap 5 detik, target lain setiap 15 detik:

| Job | Target (interval) | Isi |
| ----------- | ----------- | ----------- |
| `churn-gateway` | `34.122.18.16:8080/metrics` | Metrik aplikasi gateway: jumlah dan latensi request, prediksi per kelas, sebaran probabilitas, error, status model |
| `tf-serving` | `34.122.18.16:8080/monitoring/prometheus/metrics` (5 detik) | Metrik bawaan TensorFlow Serving yang diambil dari endpoint **cloud** (`:tensorflow:serving:request_count`, latensi, status load model) |
| `prometheus` | `127.0.0.1:9090` | Metrik Prometheus sendiri |

Metrik kustom pada gateway:

| Metrik | Jenis | Kegunaan |
| ----------- | ----------- | ----------- |
| `churn_http_requests_total{endpoint,method,status}` | Counter | Laju request dan kode status |
| `churn_request_latency_seconds` | Histogram | Latensi request, dipakai untuk p50/p95 |
| `churn_tfserving_latency_seconds` | Histogram | Latensi pemanggilan TensorFlow Serving |
| `churn_predictions_total{label}` | Counter | Jumlah prediksi churn dan tidak churn |
| `churn_prediction_probability` | Histogram | Sebaran probabilitas churn (untuk mendeteksi pergeseran) |
| `churn_errors_total{type}` | Counter | Error validasi, bad request, dan kegagalan TensorFlow Serving |
| `churn_model_available`, `churn_model_version` | Gauge | Status dan versi model aktif |
| `churn_requests_in_progress` | Gauge | Request yang sedang diproses |

### Hasil monitoring

Sistem diuji dengan lalu lintas campuran (prediksi tunggal, prediksi batch 5-40 pelanggan, serta data tidak valid dan bukan JSON untuk memicu error). Jumlah request dan prediksi adalah total sejak layanan gateway dijalankan; laju dan latensi dihitung pada jendela 15 menit terakhir. Hasil yang terbaca di Prometheus:

| Indikator | Nilai |
| ----------- | ----------- |
| Target `churn-gateway`, `tf-serving`, `prometheus` | Seluruhnya **UP** (ketersediaan gateway 100,0%) |
| Request `/predict` sukses (200) | 364 |
| Request ditolak: data tidak valid (422) | 31 |
| Request ditolak: bukan JSON atau jumlah salah (400) | 17 |
| Total prediksi | 1.552 (churn 366, tidak churn 1.186) |
| Rata-rata probabilitas churn | 0,284 |
| Laju request `/predict` (rata-rata jendela 15 menit) | 0,46 request/detik |
| Latensi `/predict` p50 / p95 | 6,4 ms / 123,1 ms |
| Latensi rata-rata TensorFlow Serving | 8,2 ms |
| `churn_model_available` | 1 (model `AVAILABLE`, versi 1793092663) |

Interpretasi: seluruh komponen sehat dan model selalu tersedia. Error yang tercatat berasal dari request uji yang sengaja tidak valid (kode 422 dan 400), bukan dari kegagalan model; tidak ada error pada TensorFlow Serving. Latensi p95 pada jendela 15 menit terakhir hanya 123,1 ms; pada grafik per 1 menit (`screenshots/3_prometheus_latency_p95.png`) p95 mulai sekitar 0,8 detik pada menit pertama (pemanasan), naik ke sekitar 0,1-0,33 detik saat request batch (hingga 40 pelanggan) masuk, dan berada di bawah 0,03 detik di luar itu. VM e2-micro memadai untuk model sekecil ini, tetapi batch besar akan menjadi pemicu pertama bila beban naik. Proporsi prediksi churn (23,6%) sedikit di bawah proporsi churn pada data latih (26,54%), wajar karena recall model sekitar 56% dan lalu lintas uji diambil dari dataset yang sama. Bila sebaran probabilitas atau proporsi prediksi bergeser jauh dari nilai ini pada lalu lintas nyata, itu menjadi sinyal awal *data drift* yang perlu diselidiki; metrik `churn_prediction_probability` disediakan untuk keperluan itu.

Screenshot dashboard monitoring: `fauzi19-monitoring.png` (halaman Prometheus pada browser). Tangkapan layar pendukung tersimpan di folder `screenshots/`:

- `screenshots/1_prometheus_targets.png`
- `screenshots/2_prometheus_request_rate.png`
- `screenshots/3_prometheus_latency_p95.png`
- `screenshots/4_prometheus_predictions_per_class.png`
- `screenshots/5_prometheus_errors.png`
- `screenshots/6_prometheus_model_available.png`
- `screenshots/7_prometheus_tfserving_requests.png`
- `screenshots/8_webapp_prediksi_churn.png`
